# Research QuantBook: ForexCarry (G10 FX Momentum)

## Objectif
Reproduire l'analyse exploratoire de `research.ipynb` avec les donnees natives QuantConnect.

## Performance actuelle
- **Sharpe**: -0.324, **CAGR**: 1.5%, **MaxDD**: 12.3%
- **Signal**: Composite momentum 21d (0.7) + 126d (0.3), risk-adjusted
- **Univers**: EURUSD, AUDUSD, USDJPY, USDCAD
- **Selection**: Top-2, long-only, monthly rebal

## Hypotheses a tester
1. Pure vs risk-adjusted momentum
2. Skip-month effect for FX
3. Lookback period (21d, 63d, 126d, 252d)
4. Number of positions (1, 2, 3)
5. Universe expansion (add GBPUSD, NZDUSD)

## Prerequis
- Environnement Lean Research
- Duree estimee: ~5 minutes

In [ ]:
# Setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

qb = QuantBook()
print("QuantBook initialise.")

## 1. Chargement des donnees

4 paires actuelles + 2 candidates supplementaires.

In [ ]:
fx_pairs = ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD', 'GBPUSD', 'NZDUSD']

symbols = {}
for pair in fx_pairs:
    try:
        symbols[pair] = qb.add_forex(pair, Resolution.DAILY).symbol
    except:
        print(f"Warning: {pair} non disponible")

start = datetime(2010, 1, 1)
end = datetime(2026, 1, 1)

history = qb.history(list(symbols.values()), start, end, Resolution.DAILY)
closes = history['close'].unstack(level=0)

stm = {str(v): k for k, v in symbols.items()}
closes.columns = [stm.get(str(c), str(c)) for c in closes.columns]
closes = closes.dropna()

print(f"Periode: {closes.index[0].date()} a {closes.index[-1].date()}")
print(f"Donnees: {len(closes)} jours de trading")
print(f"Paires: {list(closes.columns)}")

returns_df = closes.pct_change()

### Note sur les paires USD/XXX

Pour USDJPY et USDCAD, un prix qui monte = USD fort = devise locale faible.
Pour mesurer le momentum de la devise (pas du USD), on inverse le return.

In [ ]:
# Statistiques par paire
print(f"{'Paire':<10} {'Rend. Ann.':>12} {'Volatilite':>12} {'Sharpe':>8}")
print("-" * 45)

for pair in closes.columns:
    ret = (closes[pair].iloc[-1] / closes[pair].iloc[0]) ** (252 / len(closes)) - 1
    vol = returns_df[pair].std() * np.sqrt(252)
    sharpe = ret / vol if vol > 0 else 0
    print(f"{pair:<10} {ret:>11.1%} {vol:>11.1%} {sharpe:>7.2f}")

## 2. Fonctions de backtest

In [ ]:
def fx_momentum_score(closes, pair, lookback_short=21, lookback_long=126,
                       w_short=0.7, w_long=0.3, risk_adjusted=True,
                       skip_month=True):
    """Compute composite FX momentum score."""
    prices = closes[pair]
    
    # Invert USD/XXX pairs (USD strength = negative momentum for the currency)
    invert = pair.startswith('USD')
    
    if skip_month:
        # Skip last 21 days (Jegadeesh)
        ret_long = prices.shift(21) / prices.shift(lookback_long) - 1
    else:
        ret_long = prices / prices.shift(lookback_long) - 1
    
    ret_short = prices / prices.shift(lookback_short) - 1
    
    if invert:
        ret_long = -ret_long
        ret_short = -ret_short
    
    # Composite score
    score = w_short * ret_short + w_long * ret_long
    
    if risk_adjusted:
        vol = returns_df[pair].rolling(63).std() * np.sqrt(252)
        score = score / vol.clip(lower=0.01)
    
    return score

def backtest_fx_momentum(closes, universe, top_n=2, lookback_short=21,
                          lookback_long=126, risk_adjusted=True,
                          skip_month=True):
    """Backtest FX momentum strategy."""
    returns_df = closes.pct_change()
    
    # Compute scores for all pairs
    scores = pd.DataFrame(index=closes.index)
    for pair in universe:
        if pair in closes.columns:
            scores[pair] = fx_momentum_score(
                closes, pair, lookback_short, lookback_long,
                risk_adjusted=risk_adjusted, skip_month=skip_month
            )
    
    # Monthly rebalance
    monthly = closes.resample('MS').first().index
    start_idx = max(lookback_long + 21, 63) + 1
    
    port_ret = pd.Series(0.0, index=closes.index)
    current_pos = []
    
    for date in monthly:
        if date not in closes.index:
            continue
        if date < closes.index[start_idx]:
            continue
        
        s = scores.loc[date].dropna()
        if len(s) == 0:
            continue
        
        # Top-N with positive IR
        top = s.nlargest(top_n)
        current_pos = [p for p in top.index if top[p] > 0]
    
    # Actually compute returns with proper rebalancing
    positions = {}
    for date in monthly:
        if date not in closes.index or date < closes.index[start_idx]:
            continue
        s = scores.loc[date].dropna()
        if len(s) == 0:
            positions[date] = []
            continue
        top = s.nlargest(top_n)
        positions[date] = [p for p in top.index if top[p] > 0]
    
    current_pos = []
    weight = 0.5  # 50% per position
    
    for i in range(start_idx, len(closes)):
        date = closes.index[i]
        # Check if rebalance day
        if date in positions:
            current_pos = positions[date]
        
        if len(current_pos) > 0:
            w = 1.0 / max(len(current_pos), 1)
            for pair in current_pos:
                if pair in returns_df.columns:
                    # For USD/XXX pairs, we need to invert
                    r = returns_df[pair].iloc[i]
                    if pair.startswith('USD'):
                        r = -r
                    port_ret.iloc[i] += w * r
    
    port_ret = port_ret[port_ret.index >= closes.index[start_idx]]
    vals = (1 + port_ret).cumprod()
    total = vals.iloc[-1] - 1
    years = len(port_ret) / 252
    cagr = (1 + total) ** (1 / years) - 1 if years > 0 else 0
    vol = port_ret.std() * np.sqrt(252)
    sharpe = (cagr - 0.03) / vol if vol > 0.001 else 0
    max_dd = ((vals - vals.expanding().max()) / vals.expanding().max()).min()
    
    return {'sharpe': sharpe, 'cagr': cagr, 'max_dd': max_dd, 'vol': vol, 'cum': vals}

print("Fonctions definies.")

## 3. Hypothese 1: Pure vs risk-adjusted momentum

Regle #1 du backlog: risk-adjusted momentum > raw momentum.

In [ ]:
base_universe = ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD']

print(f"{'Momentum Type':<25} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8}")
print("-" * 50)

results_mom = {}
for name, ra in [('Pure momentum', False), ('Risk-adjusted (actuel)', True)]:
    r = backtest_fx_momentum(closes, base_universe, risk_adjusted=ra)
    results_mom[name] = r
    print(f"{name:<25} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%}")

### Verdict H1

research.ipynb H1 etait inconclusif sur FX. Verifier si risk-adjustment
aide ou nuit specifiquement sur les paires FX G10.

## 4. Hypothese 2: Skip-month effect

In [ ]:
print(f"{'Skip-Month':<25} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8}")
print("-" * 50)

results_skip = {}
for name, skip in [('Sans skip (J-126 a J-0)', False), ('Avec skip (J-126 a J-21)', True)]:
    r = backtest_fx_momentum(closes, base_universe, skip_month=skip)
    results_skip[name] = r
    print(f"{name:<25} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%}")

### Verdict H2

Regle #2 du backlog: skip-month confirme pour equities (Jegadeesh 1990).
Pour FX, l'effet est moins documente. Verifier sur donnees QC.

## 5. Hypothese 3: Lookback period

In [ ]:
print(f"{'Lookback':<20} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8}")
print("-" * 45)

results_lb = {}
for lb_name, lb_d in [('21d (1m)', 21), ('63d (3m)', 63), ('126d (6m, actuel)', 126), ('252d (12m)', 252)]:
    r = backtest_fx_momentum(closes, base_universe, lookback_long=lb_d)
    results_lb[lb_name] = r
    print(f"{lb_name:<20} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%}")

### Verdict H3

126d (6 mois) est le lookback long actuel. FX momentum est typiquement
plus court que equity momentum (1-3 mois vs 12 mois).

## 6. Hypothese 4: Top-N positions

In [ ]:
print(f"{'Top-N':<15} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8}")
print("-" * 40)

results_topn = {}
for n in [1, 2, 3]:
    r = backtest_fx_momentum(closes, base_universe, top_n=n)
    name = f'Top-{n}'
    results_topn[name] = r
    print(f"{name:<15} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%}")

### Verdict H4

Top-2 (actuel) = 50% par position. Top-1 = 100% concentration.
Top-3 = 33% mais sur 4 paires = presque equal weight.

## 7. Hypothese 5: Universe expansion

In [ ]:
universes = {
    '4 paires (actuel)': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD'],
    '+ GBPUSD': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD', 'GBPUSD'],
    '+ NZDUSD': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD', 'NZDUSD'],
    '6 paires': ['EURUSD', 'AUDUSD', 'USDJPY', 'USDCAD', 'GBPUSD', 'NZDUSD'],
}

print(f"{'Univers':<25} {'Sharpe':>8} {'CAGR':>8} {'MaxDD':>8}")
print("-" * 50)

results_univ = {}
for name, univ in universes.items():
    avail = [p for p in univ if p in closes.columns]
    r = backtest_fx_momentum(closes, avail)
    results_univ[name] = r
    print(f"{name:<25} {r['sharpe']:>8.3f} {r['cagr']:>7.1%} {r['max_dd']:>7.1%}")

### Verdict H5

iter5 a teste 6 paires + top-3 + USD trend filter -> Sharpe -0.849.
Plus de paires ne sauve pas le momentum FX si le probleme est structurel.

## 8. Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

ax = axes[0, 0]
for name, r in results_mom.items():
    ax.plot(r['cum'].values, label=f"{name} (S={r['sharpe']:.2f})", linewidth=1.5)
ax.set_title('H1: Pure vs Risk-adjusted momentum', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
for name, r in results_lb.items():
    ax.plot(r['cum'].values, label=f"{name} (S={r['sharpe']:.2f})", linewidth=1.5)
ax.set_title('H3: Lookback period', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
for name, r in results_topn.items():
    ax.plot(r['cum'].values, label=f"{name} (S={r['sharpe']:.2f})", linewidth=1.5)
ax.set_title('H4: Top-N positions', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
for name, r in results_univ.items():
    ax.plot(r['cum'].values, label=f"{name} (S={r['sharpe']:.2f})", linewidth=1.5)
ax.set_title('H5: Universe expansion', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('forexcarry_quantbook_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Conclusions

### Tableau recapitulatif

| Hypothese | Resultat QuantBook | Coherent avec yfinance? |
|-----------|-------------------|-------------------------|
| H1 Risk-adjusted | (a remplir) | (a verifier) |
| H2 Skip-month | (a remplir) | (a verifier) |
| H3 Lookback | (a remplir) | (a verifier) |
| H4 Top-N | (a remplir) | (a verifier) |
| H5 Universe | (a remplir) | (a verifier) |

### Probleme structurel

FX momentum G10 genere ~0.7-1.5% CAGR absolu, mais le taux sans risque
est de 2.5%-5.5% sur 2015-2026. Le Sharpe negatif est donc structurel:
le momentum FX ne compense pas le cout d'opportunite du cash.

Menkhoff et al. (2012) documentent l'affaiblissement du FX momentum post-2008,
lie a l'intervention massive des banques centrales.